# Introduction to Regression

In this notebook, I will provide an introduction to **Regression**.

A **Regression problem** is a type of machine learning task associated with predicting a **numerical, continuous value**. For example, predicting the temperature of a room given historical data that we have previously collected and recorded. 

Remember, this can be thought of as a statistical inference problem where we assume our data points are **i.i.d.** (Independent and Identically Distributed).

---

## The Mathematics of Linear Regression

In Linear Regression, we assume a linear relationship between the input variables ($x$) and the single output variable ($y$).

### 1. The Model Hypothesis
The simplest form of the model is defined by the linear equation:

$$y = wx + b$$

Where:
* $y$ is the **predicted value** (dependent variable).
* $x$ is the **feature** (independent variable).
* $w$ is the **weight** (slope/gradient).
* $b$ is the **bias** (y-intercept).

### 2. The Loss Function (Mean Squared Error)
To train the model, we need to measure how "wrong" our predictions are. We use the **Mean Squared Error (MSE)**, which calculates the average of the squared differences between the actual values ($y_i$) and the predicted values ($\hat{y}_i$):

$$J(w, b) = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$



### 3. Optimization via Gradient Descent
To find the best values for $w$ and $b$, we minimize the loss function $J$ by taking its derivative with respect to each parameter and updating them iteratively:

$$w := w - \alpha \frac{\partial J}{\partial w}$$
$$b := b - \alpha \frac{\partial J}{\partial b}$$

Where $\alpha$ (alpha) is the **learning rate**, controlling the size of the steps we take toward the minimum loss.



---

## Simulation Example: Linear Relationship with Noise

Let’s walk through an example. Suppose we have a "ground truth" relationship defined by the linear equation:

$$y = 2x + 10$$

In this scenario, the parameters are $w = 2$ and $b = 10$. This function allows us to predict a corresponding $y$ value for any arbitrary point $x$. 

However, real-world data is rarely perfect. To simulate realistic observations, we add **Gaussian noise** ($\epsilon$) to our relationship:

$$y = 2x + 10 + \epsilon$$

Where $\epsilon \sim \mathcal{N}(\mu, \sigma^2)$. Our goal is to simulate this data across a range of $x$ from $-20$ to $20$ and observe how the distribution of our data changes as we adjust the **mean ($\mu$)** and **variance ($\sigma^2$)** of the noise.

In [26]:
import random
random.seed(42)
def linear_function(a, b, x, mu, sigma):
    """this function takes arguments a and b as the parameters of a linear function and also data x"""
    ran_noise = random.gauss(mu=mu, sigma=sigma)
    y = a*x +b + ran_noise
    return y

def get_y_values(x_values, b, a, mu, sigma):
    y_values = []
    for x_value in x_values:
        y = linear_function(a,b,x_value, mu, sigma)
        y_values.append(y)
    return y_values


In [27]:
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact
import ipywidgets as widgets

x_values = np.arange(-20,20,0.5) #Data points
b = 10
a = 2
@interact(mu=(0.0, 5, 0.5), sigma=(1.0, 10, 1))
def plot_data(mu=0.0, sigma=1.0):
    y_values = get_y_values(x_values, b , a, mu=mu, sigma=sigma)
    plt.figure(figsize=(8, 5))
    plt.plot(x_values, y_values, "bo")
    plt.title("Sample data for regression with mean {} and variance {}".format(mu, sigma))
    plt.xlabel("X")
    plt.ylabel("Y")
    plt.grid(True)
    plt.ylim(-20, 60)
    plt.show()





interactive(children=(FloatSlider(value=0.0, description='mu', max=5.0, step=0.5), FloatSlider(value=1.0, desc…

 * As you can see, when the mean and variance of the noise increase, the relationship between $x$ and $y$ appears increasingly non-linear and scattered. 

 * Despite this added noise, we can use linear regression to estimate the underlying function across different mean and variance values. 
 * We will then compare our model's estimated parameters against the theoretical true values of our base equation (where the weight is 2 and the bias is -10).

In [ ]:
#We can write our own optimization or use the optimize from scipy
from scipy.optimize import minimize

nb_samples = len(x_values)
@interact(mu=(0.0, 5, 0.5), sigma=(1.0, 10, 1))
def plot_predition(mu=0.0, sigma=1.0):

    def loss_fn(v): # v is a list of these two paramters that we want to estimate that is w and b
        e = 0.0
        for i in range(nb_samples):
            e = e + np.square(v[0]*x_values[i] + v[1] - y_values[i])
        mse = 0.5* e
        return mse

    def gradient(v):
        g = np.zeros(shape=2)
        for i in range(nb_samples):
            g[0]= g[0] + (v[0]*x_values[i] + v[1] - y_values[i])*x_values[i]
            g[1]= g[1] + (v[0]*x_values[i] + v[1] - y_values[i])
        return g

    y_values = get_y_values(x_values, b , a, mu=mu, sigma=sigma)
    y = minimize(fun=loss_fn, x0=[0.0,0.0], jac=gradient, method='L-BFGS-B')
    parameter_m = y.x[0] # this is the gradient that is the value that is muyltiplied to x
    parameter_c = y.x[1] # this is the y intercept
    y_values_pred = get_y_values(x_values, parameter_c , parameter_m, mu=0.0, sigma=0.0)

    plt.figure(figsize=(8, 5))
    plt.plot(x_values, y_values, "bo")
    plt.plot(x_values, y_values_pred,  "red")
    param_text = f"w = {parameter_m:.2f}\nb = {parameter_c:.2f}"
    box_style = dict(boxstyle="round,pad=0.5", facecolor="white", edgecolor="gray", alpha=0.8)
    plt.text(
        x=0.05, y=0.95,
        s=param_text,
        transform=plt.gca().transAxes,
        fontsize=12,
        verticalalignment="top",
        bbox=box_style   )
    plt.title("Sample data plot with prediction for mean {}  and variance {}".format(mu, sigma))
    plt.xlabel("X")
    plt.ylabel("Y")
    plt.grid(True)
    plt.ylim(-20, 60)
    plt.show()

interactive(children=(FloatSlider(value=0.0, description='mu', max=5.0, step=0.5), FloatSlider(value=1.0, desc…

* As you can see, when the variance and mean of the noise increase, the clear linear relationship between the data points begins to erode and shift. This added noise obscures the underlying pattern, causing our model's estimated parameters to deviate from their true theoretical values.

* For the case of added noise with a mean of 0 and a variance of 3, the predictor function estimated by our model is $y = 1.98x + 10.09$. 

* If we test this with a value of, let's say, $x = 10$:
* The **predicted** $y$ is $1.98(10) + 10.09 = 29.89$.
* The **actual** $y$ (from our true base equation $y = 2x + 10$) is $30$. 

* The absolute error here is $0.11$ ($|30 - 29.89|$), which translates to a prediction accuracy of approximately 99.63% for this specific point.

* The plot below shows how the error scales as the variance of the noise increases.



In [ ]:
#We can write our own optimization or use the optimize from scipy
from scipy.optimize import minimize

nb_samples = len(x_values)
@interact(mu=(0.0, 5, 0.5), sigma=(1.0, 100, 1))
def plot_predition(mu=0.0, sigma=1.0):

    def loss_fn(v): # v is a list of these two paramters that we want to estimate that is w and b
        e = 0.0
        for i in range(nb_samples):
            e = e + np.square(v[0]*x_values[i] + v[1] - y_values[i])
        mse = 0.5* e
        return mse

    def gradient(v):
        g = np.zeros(shape=2)
        for i in range(nb_samples):
            g[0]= g[0] + (v[0]*x_values[i] + v[1] - y_values[i])*x_values[i]
            g[1]= g[1] + (v[0]*x_values[i] + v[1] - y_values[i])
        return g
    def accuracy(y_values_pred, y_actual_value: int = 30):
        accuracy = 0.0
        for i in range(nb_samples):
            error = y_values_pred[i]-y_actual_value
            percent_error = (100 - (np.abs(error)/30))
            accuracy += percent_error
        return accuracy/nb_samples

    y_values = get_y_values(x_values, b , a, mu=mu, sigma=sigma)
    y = minimize(fun=loss_fn, x0=[0.0,0.0], jac=gradient, method='L-BFGS-B')
    parameter_m = y.x[0] # this is the gradient that is the value that is muyltiplied to x
    parameter_c = y.x[1] # this is the y intercept
    y_values_pred = get_y_values(x_values, parameter_c , parameter_m, mu=0.0, sigma=0.0)
    avg_acc = accuracy(y_values_pred)


    plt.figure(figsize=(8, 5))
    plt.plot(x_values, y_values, "bo")
    plt.plot(x_values, y_values_pred,  "red")
    param_text = f"w = {parameter_m:.2f}\nb = {parameter_c:.2f}\navg_accuarcy = {avg_acc:.2f}"
    box_style = dict(boxstyle="round,pad=0.5", facecolor="white", edgecolor="gray", alpha=0.8)
    plt.text(
        x=0.05, y=0.95,
        s=param_text,
        transform=plt.gca().transAxes,
        fontsize=12,
        verticalalignment="top",
        bbox=box_style   )
    plt.title("Sample data plot with prediction for mean {}  and variance {}".format(mu, sigma))
    plt.xlabel("X")
    plt.ylabel("Y")
    plt.grid(True)
    plt.ylim(-20, 60)
    plt.show()

interactive(children=(FloatSlider(value=0.0, description='mu', max=5.0, step=0.5), FloatSlider(value=1.0, desc…

* We have implemented a basic example with only two dimensions (a single feature $x$ and a target $y$), but real-world datasets are rarely this simple. 

* When dealing with higher-dimensional data, a crucial first step is **feature engineering** and **dimensionality reduction**. 
* Real-world data often contains a mix of features: some with high entropy (high variance and information content) and others with low entropy (constant or redundant data that offers little value). 
* The goal is to filter out the noise and isolate the features that provide the highest information gain, ensuring the algorithm has the best possible inputs to predict correctly.